# FASDD_CV — YOLO11m-seg: EDA + Preprocessing + Resume Training

**Pipeline:** `labels_seg_refined/labels_seg_refined` (SAM2 + MaskRefine-v2 labels) → Comprehensive EDA → Preprocessing → YOLO11m-seg Resume/Training

**Sections:**
- **S0.** Configuration *(only cell you need to edit)*
- **S1.** Experiment Tracking — wandb (L1) · Config snapshot (L2) · Registry CSV (L3)
- **S2.** Environment Setup
- **S3.** Restore Working Directory (FASDD_CV · labels_seg_refined · labels_fixed · splits)
- **S4.** Load GT Annotations
- **S5.** Comprehensive EDA on Seg Masks
- **S6.** Preprocessing → `labels_seg_clean` (dedup + degenerate removal)
- **S7.** Build YAML + Symlinks for Seg Training
- **S8.** YOLO11m-seg Resume/Fresh Training
- **S9.** Post-Training Logging
- **S10.** Evaluation on Test Split
- **S11.** Inference Spot-Check
- **S12.** Save Checkpoints


## S0. Configuration
Edit **only this cell**. Everything else derives from it.

In [ ]:
# =============================================================================
#  S0 — CONFIGURATION  (only cell you need to edit)
# =============================================================================
import os
from datetime import datetime
from pathlib import Path

# ── Kaggle dataset paths ──────────────────────────────────────────────────────
# kayempee → ctcaothanhbang (account rename)
BASE_DIR = "/kaggle/input/datasets/ctcaothanhbang/dataset-v2/FASDD_CV/FASDD_CV"

# Correct refined segmentation labels directory.
# This notebook uses this directory directly as raw seg labels, then writes cleaned labels to /kaggle/working/labels_seg_clean.
LABELS_SEG_SOURCE = Path("/kaggle/input/datasets/ctcaothanhbang/storage/labels_seg_refined/labels_seg_refined")

# Dataset chứa labels_fixed + train/val/test yamls (từ detection baseline)
TRAINING_DATA_INPUT = Path("/kaggle/input/fasdd-training-data")

# ── Model ─────────────────────────────────────────────────────────────────────
TRAIN_MODE     = "finetune"        # "finetune" | "scratch"
YOLO_SIZE      = "m"
SEG_MODEL_INIT = f"yolo11{YOLO_SIZE}-seg.pt"

# ── Resume checkpoint ─────────────────────────────────────────────────────────
# True  → copy run folder from /kaggle/input to /kaggle/working and resume from weights/last.pt.
# False → start a fresh run from SEG_MODEL_INIT.
RESUME_TRAINING = True

RESUME_RUN_INPUT = Path(
    "/kaggle/input/datasets/ctcaothanhbang/storage/fasdd_cv/fasdd_cv/"
    "yolo11m_seg_finetune_imgsz1024_20260613-2"
)
RESUME_LAST_PT = RESUME_RUN_INPUT / "weights" / "last.pt"
RESUME_BEST_PT = RESUME_RUN_INPUT / "weights" / "best.pt"

# ── Training hyperparams (follow project-phase1-vdt.ipynb) ───────────────────
# For true resume, Ultralytics continues optimizer/LR/epoch state from last.pt.
# EPOCHS is still written into args.yaml before resume so you can extend total epochs if needed.
IMGSZ         = 1024
EPOCHS        = 100
BATCH         = 16       # dual T4: ~8/GPU at imgsz=1024
WORKERS       = 4
LR0           = 0.01
LRF           = 0.01
USE_MULTI_GPU = True
SEED          = 42

# ── Augmentation (EDA-derived — identical to detection baseline) ──────────────
AUGMENT_HSV_H     = 0.015
AUGMENT_HSV_S     = 0.700
AUGMENT_HSV_V     = 0.400
AUGMENT_FLIPUD    = 0.0
AUGMENT_DEGREES   = 0.0
AUGMENT_TRANSLATE = 0.2   # smoke center-X bias fix
AUGMENT_SCALE     = 0.5

# ── Experiment tracking ───────────────────────────────────────────────────────
WANDB_PROJECT  = "fasdd-cv"
WANDB_ENABLED  = True
EXPERIMENT_NOTES = (
    "Seg: resume yolo11m-seg imgsz=1024 from uploaded last.pt. "
    "Labels: labels_seg_refined directory. "
    "Full EDA + dedup/degenerate preprocess applied."
)

# ── Preprocessing ─────────────────────────────────────────────────────────────
MIN_BBOX_DIM              = 1e-4   # degenerate detection bbox filter
MIN_SEG_POINTS            = 3      # polygon must have >= this many points
BBOX_FALLBACK_FOR_MISSING = False  # True → create rect polygon for images with no seg

# ── Derived (do not edit) ─────────────────────────────────────────────────────
DATE_STR       = datetime.now().strftime("%Y%m%d")
FRESH_RUN_NAME = f"yolo11{YOLO_SIZE}_seg_{TRAIN_MODE}_imgsz{IMGSZ}_{DATE_STR}"
RESUME_RUN_NAME = RESUME_RUN_INPUT.name
RUN_NAME       = RESUME_RUN_NAME if RESUME_TRAINING else FRESH_RUN_NAME

WORK_DIR    = Path("/kaggle/working")
PROJECT_DIR = WORK_DIR / "fasdd_cv"
RUN_DIR     = PROJECT_DIR / RUN_NAME

LABELS_SEG_RAW   = LABELS_SEG_SOURCE             # read-only source labels from Kaggle input
LABELS_SEG_CLEAN = WORK_DIR / "labels_seg_clean" # after preprocessing (train)
LABELS_FIXED     = WORK_DIR / "labels_fixed"     # GT bbox labels (reference)
YAML_PATH        = WORK_DIR / "fasdd_cv_seg.yaml"

BASE_DIR_PATH = Path(BASE_DIR)
IMG_DIR       = BASE_DIR_PATH / "images"
LABEL_DIR     = BASE_DIR_PATH / "annotations" / "YOLO_CV" / "labels"
YOLO_ANN_DIR  = BASE_DIR_PATH / "annotations" / "YOLO_CV"
TRAIN_TXT     = YOLO_ANN_DIR / "train.txt"
VAL_TXT       = YOLO_ANN_DIR / "val.txt"
TEST_TXT      = YOLO_ANN_DIR / "test.txt"

TRAIN_LIST = WORK_DIR / "train_images.txt"
VAL_LIST   = WORK_DIR / "val_images.txt"
TEST_LIST  = WORK_DIR / "test_images.txt"

WORKING_IMAGES = WORK_DIR / "images"
WORKING_LABELS = WORK_DIR / "labels"

# Only create writable working directories here. Do not mkdir inside /kaggle/input.
for d in [LABELS_SEG_CLEAN, PROJECT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("FASDD_CV — YOLO11m-seg Seg Training")
print("=" * 65)
print(f"  Run name          : {RUN_NAME}")
print(f"  Resume training   : {RESUME_TRAINING}")
if RESUME_TRAINING:
    print(f"  Resume run input  : {RESUME_RUN_INPUT}")
    print(f"  Resume last.pt    : {RESUME_LAST_PT}")
print(f"  Model init        : {SEG_MODEL_INIT}")
print(f"  IMGSZ             : {IMGSZ}")
print(f"  EPOCHS            : {EPOCHS}  BATCH: {BATCH}  LR0: {LR0}")
print(f"  Multi-GPU         : {USE_MULTI_GPU}")
print(f"  LABELS_SEG_SOURCE : {LABELS_SEG_SOURCE}")
print(f"  WANDB             : {'enabled' if WANDB_ENABLED else 'disabled'}")
print("=" * 65)


## S1. Experiment Tracking

Three layers (identical to detection baseline `project-phase1-vdt.ipynb`):
- **Layer 1 — wandb**: real-time metric streaming
- **Layer 2 — config snapshot**: `run_config.json` per run
- **Layer 3 — registry CSV**: one row per run, accumulates all experiments


In [ ]:
!pip install ultralytics wandb --quiet
print("deps ready")


In [ ]:
# =============================================================================
#  LAYER 1 — Weights & Biases
# =============================================================================
import wandb

if WANDB_ENABLED:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
        wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
        print("wandb: logged in via Kaggle Secret")
    except Exception as e:
        print(f"Kaggle Secret not found ({e}) → offline mode")
        os.environ["WANDB_MODE"] = "offline"
        wandb.login(anonymous="allow")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("wandb: disabled")


In [ ]:
# =============================================================================
#  LAYER 2 — Config Snapshot
# =============================================================================
import json as _json

def save_run_config(run_dir: Path, extra: dict = None) -> Path:
    run_dir.mkdir(parents=True, exist_ok=True)
    cfg = {
        "run_name"       : RUN_NAME,
        "date"           : DATE_STR,
        "train_mode"     : TRAIN_MODE,
        "notes"          : EXPERIMENT_NOTES,
        "task"           : "segment",
        "model_family"   : "yolo11",
        "model_size"     : YOLO_SIZE,
        "model_init"     : SEG_MODEL_INIT,
        "resume_training": RESUME_TRAINING,
        "resume_run_input": str(RESUME_RUN_INPUT) if RESUME_TRAINING else None,
        "resume_last_pt"  : str(RESUME_LAST_PT) if RESUME_TRAINING else None,
        "dataset"        : "FASDD_CV",
        "imgsz"          : IMGSZ,
        "split"          : "official (train=47660 val=31770 test=15884)",
        "epochs"         : EPOCHS,
        "batch"          : BATCH,
        "workers"        : WORKERS,
        "lr0"            : LR0,
        "lrf"            : LRF,
        "warmup_epochs"  : 3.0,
        "mosaic"         : 1.0,
        "weight_decay"   : 0.0005,
        "patience"       : 50,
        "use_multi_gpu"  : USE_MULTI_GPU,
        "seed"           : SEED,
        "hsv_h"          : AUGMENT_HSV_H,
        "hsv_s"          : AUGMENT_HSV_S,
        "hsv_v"          : AUGMENT_HSV_V,
        "flipud"         : AUGMENT_FLIPUD,
        "degrees"        : AUGMENT_DEGREES,
        "translate"      : AUGMENT_TRANSLATE,
        "scale"          : AUGMENT_SCALE,
        "label_source"   : str(LABELS_SEG_SOURCE),
        "bbox_fallback"  : BBOX_FALLBACK_FOR_MISSING,
        "results": {
            # detection head
            "box_mAP50": None, "box_mAP50_fire": None,
            "box_mAP50_smoke": None, "box_mAP5095": None,
            # segmentation head
            "seg_mAP50": None, "seg_mAP50_fire": None,
            "seg_mAP50_smoke": None, "seg_mAP5095": None,
            "epochs_run": None, "inference_ms": None, "model_mb": None,
        }
    }
    if extra:
        cfg.update(extra)
    cfg_path = run_dir / "run_config.json"
    with open(cfg_path, "w") as f:
        _json.dump(cfg, f, indent=2)
    print(f"Config snapshot → {cfg_path}")
    return cfg_path

print("Layer 2 ready.")


In [ ]:
# =============================================================================
#  LAYER 3 — Experiment Registry CSV
# =============================================================================
import pandas as pd

REGISTRY_PATH = WORK_DIR / "experiment_registry_seg.csv"

REGISTRY_COLUMNS = [
    "run_name", "date", "model", "mode", "imgsz", "epochs_run",
    # box head
    "box_mAP50", "box_mAP50_fire", "box_mAP50_smoke", "box_mAP5095",
    # seg head
    "seg_mAP50", "seg_mAP50_fire", "seg_mAP50_smoke", "seg_mAP5095",
    "AP_gap_smoke_minus_fire_box",
    "precision", "recall",
    "inference_ms_t4", "fps_t4", "model_mb", "onnx_mb",
    "notes", "wandb_run_url",
]

def load_registry():
    if REGISTRY_PATH.exists():
        return pd.read_csv(REGISTRY_PATH)
    return pd.DataFrame(columns=REGISTRY_COLUMNS)

def append_to_registry(row: dict):
    df = load_registry()
    for col in REGISTRY_COLUMNS:
        if col not in row:
            row[col] = None
    new_row = pd.DataFrame([{c: row.get(c) for c in REGISTRY_COLUMNS}])
    df = pd.concat([df, new_row], ignore_index=True)
    df.to_csv(REGISTRY_PATH, index=False)
    print(f"Registry updated: {REGISTRY_PATH}  ({len(df)} total runs)")
    return df

def show_registry():
    df = load_registry()
    if df.empty:
        print("Registry empty — no completed runs yet.")
        return df
    key_cols = ["run_name", "model", "mode", "imgsz", "epochs_run",
                "box_mAP50", "box_mAP50_fire", "seg_mAP50", "seg_mAP50_fire",
                "inference_ms_t4", "notes"]
    try:
        from IPython.display import display
        display(df[[c for c in key_cols if c in df.columns]])
    except:
        print(df[[c for c in key_cols if c in df.columns]].to_string())
    return df

print("Layer 3 ready.")
existing = load_registry()
print(f"Existing runs in registry: {len(existing)}")
if not existing.empty:
    show_registry()


## S2. Environment Setup

In [ ]:
import random, warnings, time, zipfile, shutil
import numpy as np
import pandas as pd
import torch
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter, defaultdict
from pathlib import Path
from tqdm.auto import tqdm
import yaml

warnings.filterwarnings("ignore")

import ultralytics
print(f"ultralytics : {ultralytics.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {name} ({vram:.1f} GB)")

# Seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## S3. Restore Working Directory

### 3.1 Unzip FASDD_CV Dataset


In [ ]:
# Unzip FASDD_CV (~11 GB) nếu chưa có
def find_data_root(base="/kaggle/input"):
    for candidate in [
        f"/kaggle/input/datasets/ctcaothanhbang/dataset-v2/FASDD_CV/FASDD_CV",
        "/kaggle/input/fasdd-cv/FASDD_CV",
    ]:
        if Path(candidate).is_dir() and (Path(candidate) / "images").is_dir():
            return candidate
    return None

data_root = find_data_root()
if data_root:
    print(f"Dataset found: {data_root}")
    # Update paths if needed
    if str(BASE_DIR_PATH) != data_root:
        print(f"  NOTE: BASE_DIR in S0 is different from found path — using found path")
    IMG_DIR   = Path(data_root) / "images"
    LABEL_DIR = Path(data_root) / "annotations" / "YOLO_CV" / "labels"
    YOLO_ANN_DIR = Path(data_root) / "annotations" / "YOLO_CV"
    TRAIN_TXT = YOLO_ANN_DIR / "train.txt"
    VAL_TXT   = YOLO_ANN_DIR / "val.txt"
    TEST_TXT  = YOLO_ANN_DIR / "test.txt"
else:
    print("ERROR: FASDD_CV dataset not found — check BASE_DIR in S0")

print(f"  IMG_DIR   : {IMG_DIR}  exists={IMG_DIR.exists()}")
print(f"  LABEL_DIR : {LABEL_DIR}  exists={LABEL_DIR.exists()}")
print(f"  TRAIN_TXT : {TRAIN_TXT.exists()}")


### 3.2 Use `labels_seg_refined` Directory


In [ ]:
# Use refined seg labels directory directly.
# Expected path:
#   /kaggle/input/datasets/ctcaothanhbang/storage/labels_seg_refined/labels_seg_refined
#
# Later steps read LABELS_SEG_RAW and write cleaned training labels to:
#   /kaggle/working/labels_seg_clean

assert LABELS_SEG_RAW.exists(), (
    f"\nERROR: LABELS_SEG_RAW does not exist:\n  {LABELS_SEG_RAW}\n"
    "Check LABELS_SEG_SOURCE in S0 and make sure the Kaggle dataset is attached."
)
assert LABELS_SEG_RAW.is_dir(), (
    f"\nERROR: LABELS_SEG_RAW is not a directory:\n  {LABELS_SEG_RAW}\n"
)

flat_txt = sorted(LABELS_SEG_RAW.glob("*.txt"))

# If the uploaded dataset has an extra nested directory, flatten it into /kaggle/working/labels_seg_raw.
# For your current requested path this should usually not run, because the .txt files should already be directly inside LABELS_SEG_RAW.
if len(flat_txt) == 0:
    nested_txt = sorted(LABELS_SEG_RAW.rglob("*.txt"))
    assert len(nested_txt) > 0, (
        f"\nERROR: no .txt label files found under:\n  {LABELS_SEG_RAW}\n"
    )

    WORK_RAW = WORK_DIR / "labels_seg_raw"
    if WORK_RAW.exists() or WORK_RAW.is_symlink():
        if WORK_RAW.is_symlink() or WORK_RAW.is_file():
            WORK_RAW.unlink()
        else:
            shutil.rmtree(WORK_RAW)
    WORK_RAW.mkdir(parents=True, exist_ok=True)

    seen_names = set()
    for src in tqdm(nested_txt, desc="Flatten labels_seg_refined", ncols=80):
        if src.name in seen_names:
            raise ValueError(f"Duplicate label filename found while flattening: {src.name}")
        seen_names.add(src.name)
        shutil.copy2(src, WORK_RAW / src.name)

    LABELS_SEG_RAW = WORK_RAW
    flat_txt = sorted(LABELS_SEG_RAW.glob("*.txt"))
    print(f"Flattened nested labels into: {LABELS_SEG_RAW}")

n_seg_files = len(flat_txt)
print(f"Using segmentation labels from: {LABELS_SEG_RAW}")
print(f"labels_seg_raw/refined files : {n_seg_files:,}")

# Basic sanity checks.
assert n_seg_files > 0, "ERROR: No .txt files found in refined segmentation label directory."
example_files = flat_txt[:3]
print("Examples:")
for p in example_files:
    print(f"  {p.name}")


### 3.3 Restore `labels_fixed` + Image Split Lists

In [ ]:
# ── Option A: restore từ training-data Kaggle dataset ─────────────────────────
if TRAINING_DATA_INPUT.exists():
    if not LABELS_FIXED.exists():
        shutil.copytree(TRAINING_DATA_INPUT / "labels_fixed", LABELS_FIXED)
        print(f"Copied labels_fixed: {len(list(LABELS_FIXED.glob('*.txt'))):,} files")
    else:
        print(f"labels_fixed exists: {len(list(LABELS_FIXED.glob('*.txt'))):,} files")

    for f in ["fasdd_cv.yaml", "train_images.txt", "val_images.txt", "test_images.txt"]:
        src = TRAINING_DATA_INPUT / f
        if src.exists() and not (WORK_DIR / f).exists():
            shutil.copy(src, WORK_DIR / f)
            print(f"Copied: {f}")

# ── Option B: build from scratch using FASDD_CV annotations ──────────────────
else:
    print("TRAINING_DATA_INPUT not found — rebuilding from FASDD_CV annotations")

    def read_split_stems(txt_path: Path) -> list:
        stems = []
        if not txt_path.exists():
            return stems
        with open(txt_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    stems.append(Path(line).stem)
        return stems

    train_stems = read_split_stems(TRAIN_TXT)
    val_stems   = read_split_stems(VAL_TXT)
    test_stems  = read_split_stems(TEST_TXT)
    print(f"Splits from FASDD_CV — train: {len(train_stems):,}  val: {len(val_stems):,}  test: {len(test_stems):,}")

    WORKING_IMAGES.unlink(missing_ok=True) if WORKING_IMAGES.is_symlink() else None
    WORKING_IMAGES.symlink_to(IMG_DIR)

    def write_img_list(stems, out_path):
        with open(out_path, "w") as f:
            for stem in stems:
                f.write(str(WORKING_IMAGES / (stem + ".jpg")) + "\n")

    write_img_list(train_stems, TRAIN_LIST)
    write_img_list(val_stems,   VAL_LIST)
    write_img_list(test_stems,  TEST_LIST)
    print("Image split lists written.")

print(f"\nTRAIN_LIST: {TRAIN_LIST.exists()}  ({sum(1 for _ in open(TRAIN_LIST))} lines)")
print(f"VAL_LIST  : {VAL_LIST.exists()}  ({sum(1 for _ in open(VAL_LIST))} lines)")
print(f"TEST_LIST : {TEST_LIST.exists()}  ({sum(1 for _ in open(TEST_LIST))} lines)")


### 3.4 Fix Degenerate GT Bboxes → `labels_fixed`

In [ ]:
# Scan all .txt files, remove any line with bw < 1e-4 OR bh < 1e-4
# (same logic as project-phase1-vdt.ipynb Section 4.1)
LABELS_FIXED.mkdir(parents=True, exist_ok=True)
n_existing_fixed = len(list(LABELS_FIXED.glob("*.txt")))

if n_existing_fixed > 50_000:
    print(f"labels_fixed already built: {n_existing_fixed:,} files — skipping")
else:
    label_files = sorted(LABEL_DIR.glob("*.txt"))
    print(f"Fixing degenerate bboxes in {len(label_files):,} files...")

    n_files_touched = 0
    n_lines_removed = 0
    removed_log = []

    for lf in tqdm(label_files, desc="Fix degenerate", ncols=80):
        with open(lf) as f:
            lines = f.readlines()
        clean = []
        for line in lines:
            stripped = line.strip()
            if not stripped:
                clean.append(line)
                continue
            parts = stripped.split()
            if len(parts) != 5:
                clean.append(line)
                continue
            try:
                bw, bh = float(parts[3]), float(parts[4])
            except:
                clean.append(line)
                continue
            if bw < MIN_BBOX_DIM or bh < MIN_BBOX_DIM:
                n_lines_removed += 1
                removed_log.append(f"{lf.name}: bw={bw:.6f} bh={bh:.6f}")
            else:
                clean.append(line)
        out = LABELS_FIXED / lf.name
        with open(out, "w") as f:
            f.writelines(clean)
        if len(clean) != len(lines):
            n_files_touched += 1

    print(f"Files modified : {n_files_touched}")
    print(f"Lines removed  : {n_lines_removed}")
    for entry in removed_log:
        print(f"  {entry}")
    if n_lines_removed == 0:
        print("  (no degenerate boxes found)")
    print(f"labels_fixed   : {len(list(LABELS_FIXED.glob('*.txt'))):,} files")


## S4. Load GT Annotations

In [ ]:
# Load all GT bboxes from labels_fixed
# Format: class_id cx cy bw bh (YOLO normalized)

all_img_paths = sorted(IMG_DIR.glob("*.jpg"))
print(f"Total images in dataset: {len(all_img_paths):,}")

active_label_dir = LABELS_FIXED if LABELS_FIXED.exists() else LABEL_DIR
print(f"Label source: {active_label_dir}")

img_ann_map = {}  # stem → [(cls_id, cx, cy, bw, bh), ...]
for p in tqdm(all_img_paths, desc="Loading GT", ncols=80):
    lbl = active_label_dir / (p.stem + ".txt")
    boxes = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                try:
                    cls_id = int(parts[0])
                    cx, cy, bw, bh = map(float, parts[1:])
                    boxes.append((cls_id, cx, cy, bw, bh))
                except:
                    pass
    img_ann_map[p.stem] = boxes

ann_stems = [p.stem for p in all_img_paths if img_ann_map.get(p.stem)]
bg_stems  = [p.stem for p in all_img_paths if not img_ann_map.get(p.stem)]

n_fire  = sum(1 for v in img_ann_map.values() for b in v if b[0] == 0)
n_smoke = sum(1 for v in img_ann_map.values() for b in v if b[0] == 1)
print(f"Annotated images : {len(ann_stems):,}")
print(f"Background images: {len(bg_stems):,}")
print(f"Fire boxes       : {n_fire:,}")
print(f"Smoke boxes      : {n_smoke:,}")


## S5. Comprehensive EDA on Seg Masks

### 5.0 — Parse All Seg Files


In [ ]:
# Parse all seg files and collect metrics per polygon line and per image
print("Parsing seg files from labels_seg_raw/ ...")

seg_records  = []   # one row per polygon line
cov_records  = []   # one row per annotated image
split_map    = {}   # stem → 'train'/'val'/'test'

for split_name, list_file in [("train", TRAIN_LIST), ("val", VAL_LIST), ("test", TEST_LIST)]:
    if list_file.exists():
        for line in open(list_file):
            stem = Path(line.strip()).stem
            split_map[stem] = split_name

# Helper: compute polygon area (shoelace, normalized)
def poly_area_normalized(coords):
    xs, ys = coords[0::2], coords[1::2]
    n = len(xs)
    if n < 3:
        return 0.0
    area = abs(sum(xs[i]*ys[i+1] - xs[i+1]*ys[i] for i in range(n-1)) +
               xs[-1]*ys[0] - xs[0]*ys[-1]) / 2
    return float(area)

for stem in tqdm(ann_stems, desc="Parsing", ncols=80):
    seg_file = LABELS_SEG_RAW / (stem + ".txt")
    gt_boxes = img_ann_map[stem]
    n_gt     = len(gt_boxes)
    split    = split_map.get(stem, "unknown")

    raw_lines = []
    if seg_file.exists() and seg_file.stat().st_size > 0:
        raw_lines = [l.strip() for l in seg_file.read_text().splitlines() if l.strip()]

    n_seg = len(raw_lines)
    line_counts = Counter(raw_lines)
    n_exact_dup  = sum(c - 1 for c in line_counts.values())   # extra copies

    # Coverage status
    if   n_seg == 0:       status = "empty"
    elif n_seg == n_gt:    status = "perfect"
    elif n_seg > n_gt:     status = "over"
    else:                  status = "partial"

    cov_records.append({
        "stem": stem, "n_gt": n_gt, "n_seg": n_seg,
        "n_exact_dup": n_exact_dup, "status": status,
        "has_file": seg_file.exists(), "split": split,
    })

    # Per-polygon metrics
    for line in raw_lines:
        parts  = line.split()
        n_tok  = len(parts)
        is_dup = line_counts[line] > 1

        try:
            cls_id = int(parts[0])
        except:
            cls_id = -1

        n_coords = n_tok - 1
        n_pts    = n_coords // 2

        # Validity checks
        is_degen = (n_tok < 7) or (n_coords % 2 != 0)   # < 3 points or odd coords

        coords_ok   = False
        poly_area   = 0.0
        inside_ratio = 0.0

        if not is_degen and cls_id >= 0:
            try:
                coords = list(map(float, parts[1:]))
                coords_ok = all(0.0 <= v <= 1.0 for v in coords)
                poly_area = poly_area_normalized(coords)

                # Inside ratio: polygon centroid vs nearest GT box of same class
                xs, ys = coords[0::2], coords[1::2]
                px, py = sum(xs)/len(xs), sum(ys)/len(ys)
                for b_cls, b_cx, b_cy, b_bw, b_bh in gt_boxes:
                    if b_cls == cls_id:
                        x1, y1 = b_cx - b_bw/2, b_cy - b_bh/2
                        x2, y2 = b_cx + b_bw/2, b_cy + b_bh/2
                        if x1 <= px <= x2 and y1 <= py <= y2:
                            inside_ratio = 1.0
                            break
            except:
                pass

        seg_records.append({
            "stem": stem, "split": split, "cls_id": cls_id,
            "n_pts": n_pts, "is_degenerate": is_degen,
            "coords_ok": coords_ok, "poly_area": poly_area,
            "is_dup": is_dup, "inside_ratio": inside_ratio,
        })

df_seg = pd.DataFrame(seg_records)
df_cov = pd.DataFrame(cov_records)

print(f"Parsed {len(df_seg):,} polygon lines from {len(df_cov):,} annotated images.")


### 5.1 — Dataset-Level Statistics

In [ ]:
print("=" * 65)
print("S5.1 — DATASET-LEVEL STATISTICS")
print("=" * 65)

n_ann    = len(ann_stems)
n_total  = len(all_img_paths)
n_seg_files = df_cov["has_file"].sum()
n_tot_gt    = df_cov["n_gt"].sum()
n_tot_seg   = df_cov["n_seg"].sum()

print(f"Total images                    : {n_total:,}")
print(f"  Annotated (has GT boxes)      : {n_ann:,}  ({n_ann/n_total*100:.1f}%)")
print(f"  Background (no GT)            : {len(bg_stems):,}")
print()
print(f"Seg files present               : {n_seg_files:,}  ({n_seg_files/n_ann*100:.1f}% of annotated)")
print(f"Seg files missing               : {n_ann - n_seg_files:,}")
print()
print(f"Total GT boxes                  : {n_tot_gt:,}")
print(f"Total seg polygon lines         : {n_tot_seg:,}  ({n_tot_seg/max(1,n_tot_gt)*100:.1f}% of GT)")
print()
print(f"Polygon quality:")
n_degen  = df_seg["is_degenerate"].sum()
n_oor    = (~df_seg["coords_ok"] & ~df_seg["is_degenerate"]).sum()
n_dup    = df_seg["is_dup"].sum()
n_good   = len(df_seg) - n_degen - n_oor
print(f"  Degenerate (< 3 pts OR odd)   : {n_degen:,}  ({n_degen/max(1,len(df_seg))*100:.2f}%)")
print(f"  Coords out-of-range [0,1]     : {n_oor:,}  ({n_oor/max(1,len(df_seg))*100:.2f}%)")
print(f"  Exact duplicates (extra copy) : {n_dup:,}  ({n_dup/max(1,len(df_seg))*100:.2f}%)")
print(f"  Good polygons                 : {n_good:,}  ({n_good/max(1,len(df_seg))*100:.2f}%)")
print()

status_counts = df_cov["status"].value_counts()
print("Coverage status (images):")
for s, c in status_counts.items():
    print(f"  {s:<10}: {c:,}  ({c/n_ann*100:.1f}%)")


### 5.2 — Coverage Audit + Duplicate Analysis Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("S5.2 — Seg Coverage Audit & Duplicate Analysis", fontsize=13, fontweight="bold")
axes = axes.ravel()

STATUS_COLORS = {
    "perfect": "#27ae60", "partial": "#f39c12",
    "empty":   "#e74c3c", "over":    "#9b59b6"
}
STATUS_ORDER = ["perfect", "partial", "empty", "over"]

# ── [0] Coverage status bar ───────────────────────────────────────────────────
ax = axes[0]
sc = df_cov["status"].value_counts().reindex(STATUS_ORDER, fill_value=0)
bars = ax.bar(sc.index, sc.values,
              color=[STATUS_COLORS[k] for k in sc.index], edgecolor="white", lw=0.8)
ax.set_title("Images by Coverage Status")
ax.set_ylabel("# images")
for bar, v in zip(bars, sc.values):
    pct = v / n_ann * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f"{v:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=8)
patches = [mpatches.Patch(color=STATUS_COLORS[s], label=s) for s in STATUS_ORDER]
ax.legend(handles=patches, fontsize=7)

# ── [1] n_seg / n_gt ratio histogram ─────────────────────────────────────────
ax = axes[1]
df_ann_cov = df_cov[df_cov["n_gt"] > 0]
ratios = (df_ann_cov["n_seg"] / df_ann_cov["n_gt"]).clip(upper=2.5)
ax.hist(ratios, bins=50, color="#3498db", edgecolor="white", alpha=0.85)
ax.axvline(1.0, color="red", lw=2, ls="--", label="perfect (ratio=1.0)")
ax.set_title("n_seg / n_gt per image")
ax.set_xlabel("ratio  (clipped at 2.5)")
ax.set_ylabel("# images")
ax.legend(fontsize=8)

# ── [2] Images with exact duplicate lines ─────────────────────────────────────
ax = axes[2]
n_has_dup = (df_cov["n_exact_dup"] > 0).sum()
n_no_dup  = n_ann - n_has_dup
bars2 = ax.bar(["No duplicates", "Has duplicates"], [n_no_dup, n_has_dup],
               color=["#27ae60", "#e74c3c"], edgecolor="white")
ax.set_title("Files with exact duplicate polygon lines")
ax.set_ylabel("# images")
for bar, v in zip(bars2, [n_no_dup, n_has_dup]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f"{v:,}\n({v/n_ann*100:.1f}%)", ha="center", va="bottom", fontsize=9)

# ── [3] Distribution of total duplicate count per file ────────────────────────
ax = axes[3]
dup_counts = df_cov[df_cov["n_exact_dup"] > 0]["n_exact_dup"]
if len(dup_counts) > 0:
    ax.hist(dup_counts.clip(upper=10), bins=range(1, 12), color="#e74c3c",
            edgecolor="white", alpha=0.85, rwidth=0.85)
    ax.set_title("Duplicate line count per file\n(among files with duplicates)")
    ax.set_xlabel("# duplicate lines (clipped at 10)")
    ax.set_ylabel("# files")
    ax.set_xticks(range(1, 11))
else:
    ax.text(0.5, 0.5, "No duplicates found!", transform=ax.transAxes,
            ha="center", va="center", fontsize=12, color="green")
    ax.set_title("Duplicate line count per file")

# ── [4] n_gt_boxes per image distribution ────────────────────────────────────
ax = axes[4]
gt_counts = df_cov["n_gt"].clip(upper=10)
ax.hist(gt_counts, bins=range(1, 12), color="#2ecc71", edgecolor="white",
        alpha=0.85, rwidth=0.85)
ax.set_title("GT boxes per annotated image")
ax.set_xlabel("# GT boxes (clipped at 10)")
ax.set_ylabel("# images")
ax.set_xticks(range(1, 11))

# ── [5] Over-masked images: n_seg - n_gt ─────────────────────────────────────
ax = axes[5]
over_df = df_cov[df_cov["status"] == "over"]
if len(over_df) > 0:
    excess = (over_df["n_seg"] - over_df["n_gt"]).clip(upper=10)
    ax.hist(excess, bins=range(1, 12), color="#9b59b6", edgecolor="white",
            alpha=0.85, rwidth=0.85)
    ax.set_title(f"Over-masked: excess seg lines\n({len(over_df):,} images with n_seg > n_gt)")
    ax.set_xlabel("# excess seg lines (clipped at 10)")
    ax.set_ylabel("# files")
    ax.set_xticks(range(1, 11))
else:
    ax.text(0.5, 0.5, "No over-masked images!", transform=ax.transAxes,
            ha="center", va="center", fontsize=12, color="green")
    ax.set_title("Over-masked: excess seg lines")

plt.tight_layout()
plt.savefig(WORK_DIR / "eda_s52_coverage.png", dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved: eda_s52_coverage.png")


### 5.3 — Polygon Quality Check (Degenerate & Out-of-Range)

In [ ]:
print("=" * 65)
print("S5.3 — POLYGON QUALITY CHECK")
print("=" * 65)

# Degenerate detail
degen_df = df_seg[df_seg["is_degenerate"]]
print(f"Degenerate polygons: {len(degen_df):,}")
if len(degen_df) > 0:
    print("  Breakdown:")
    print(f"    cls_id distribution : {dict(degen_df['cls_id'].value_counts())}")
    print(f"    n_pts distribution  : {dict(degen_df['n_pts'].value_counts().head(8))}")
    # Show examples
    degen_samples = degen_df.head(5)["stem"].tolist()
    print("  Example stems:", degen_samples[:3])

# Out-of-range coords (non-degenerate polygons with coords outside [0,1])
oor_df = df_seg[~df_seg["is_degenerate"] & ~df_seg["coords_ok"]]
print(f"\nOut-of-range coords: {len(oor_df):,}")
if len(oor_df) > 0:
    print("  Example stems:", oor_df.head(3)["stem"].tolist())

# Good polygons
good_df = df_seg[~df_seg["is_degenerate"] & df_seg["coords_ok"]]
print(f"\nGood polygons: {len(good_df):,}  ({len(good_df)/max(1,len(df_seg))*100:.2f}%)")
print(f"  Fire (cls=0) : {(good_df['cls_id']==0).sum():,}")
print(f"  Smoke (cls=1): {(good_df['cls_id']==1).sum():,}")


### 5.4 — Mask Area & Polygon Point Count Distributions

In [ ]:
good = df_seg[~df_seg["is_degenerate"] & df_seg["coords_ok"]]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("S5.4 — Polygon Geometry Distributions", fontsize=13, fontweight="bold")
axes = axes.ravel()

cls_colors = {0: "#e74c3c", 1: "#3498db"}
cls_names  = {0: "fire", 1: "smoke"}

# ── [0] Polygon area distribution (log-scale) ─────────────────────────────────
ax = axes[0]
for cls_id in [0, 1]:
    subset = good[good["cls_id"] == cls_id]["poly_area"]
    if len(subset) > 0:
        ax.hist(subset.clip(upper=0.5), bins=60, alpha=0.6,
                color=cls_colors[cls_id], label=cls_names[cls_id],
                edgecolor="none", density=True)
ax.set_title("Polygon Area Distribution (normalized, clipped at 0.5)")
ax.set_xlabel("area (fraction of image)")
ax.set_ylabel("density")
ax.legend(fontsize=9)

# ── [1] Polygon area (log-scale) ──────────────────────────────────────────────
ax = axes[1]
for cls_id in [0, 1]:
    subset = good[good["cls_id"] == cls_id]["poly_area"].clip(lower=1e-6)
    if len(subset) > 0:
        ax.hist(np.log10(subset), bins=60, alpha=0.6,
                color=cls_colors[cls_id], label=cls_names[cls_id],
                edgecolor="none", density=True)
ax.set_title("Polygon Area — log₁₀ scale")
ax.set_xlabel("log₁₀(area)")
ax.set_ylabel("density")
ax.legend(fontsize=9)

# ── [2] Polygon point count distribution ──────────────────────────────────────
ax = axes[2]
for cls_id in [0, 1]:
    subset = good[good["cls_id"] == cls_id]["n_pts"].clip(upper=100)
    if len(subset) > 0:
        ax.hist(subset, bins=50, alpha=0.6,
                color=cls_colors[cls_id], label=cls_names[cls_id], edgecolor="none")
ax.axvline(MIN_SEG_POINTS, color="red", ls="--", lw=1.5, label=f"min={MIN_SEG_POINTS}")
ax.set_title("Polygon Point Count Distribution")
ax.set_xlabel("# points per polygon (clipped at 100)")
ax.set_ylabel("# polygons")
ax.legend(fontsize=9)

# ── [3] Polygon point count CDF ───────────────────────────────────────────────
ax = axes[3]
for cls_id in [0, 1]:
    subset = good[good["cls_id"] == cls_id]["n_pts"].sort_values()
    if len(subset) > 0:
        ax.plot(subset.values, np.linspace(0, 1, len(subset)),
                color=cls_colors[cls_id], label=cls_names[cls_id], lw=2)
ax.set_title("Polygon Points — CDF")
ax.set_xlabel("# points")
ax.set_ylabel("cumulative fraction")
ax.set_xlim(0, 80)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(WORK_DIR / "eda_s54_geometry.png", dpi=130, bbox_inches="tight")
plt.show()

print("Polygon point count stats:")
for cls_id in [0, 1]:
    s = good[good["cls_id"] == cls_id]["n_pts"]
    if len(s) > 0:
        print(f"  {cls_names[cls_id]}: mean={s.mean():.1f}  median={s.median():.0f}  "
              f"p5={s.quantile(0.05):.0f}  p95={s.quantile(0.95):.0f}  max={s.max()}")


### 5.5 — Per-Class & Train/Val/Test Split Breakdown

In [ ]:
print("=" * 65)
print("S5.5 — PER-CLASS & SPLIT BREAKDOWN")
print("=" * 65)

# Per-class GT vs Seg counts
for cls_id, cls_name in [(0, "fire"), (1, "smoke")]:
    n_gt_cls  = sum(1 for v in img_ann_map.values() for b in v if b[0] == cls_id)
    n_seg_cls = (df_seg["cls_id"] == cls_id).sum()
    n_good_cls = ((df_seg["cls_id"] == cls_id) &
                  ~df_seg["is_degenerate"] & df_seg["coords_ok"]).sum()
    print(f"  {cls_name:5s}: GT={n_gt_cls:,}  seg_lines={n_seg_cls:,}  "
          f"good={n_good_cls:,}  coverage={n_seg_cls/max(1,n_gt_cls)*100:.1f}%")

print()

# Per-split summary
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("S5.5 — Coverage by Split", fontsize=12, fontweight="bold")

for ax_idx, split_name in enumerate(["train", "val", "test"]):
    ax = axes[ax_idx]
    split_df = df_cov[df_cov["split"] == split_name]
    if len(split_df) == 0:
        ax.text(0.5, 0.5, f"No {split_name} data", transform=ax.transAxes,
                ha="center", va="center")
        continue

    sc = split_df["status"].value_counts().reindex(STATUS_ORDER, fill_value=0)
    bars = ax.bar(sc.index, sc.values,
                  color=[STATUS_COLORS[k] for k in sc.index], edgecolor="white")
    ax.set_title(f"{split_name}  ({len(split_df):,} images)")
    ax.set_ylabel("# images")
    n_ann_split = len(split_df)
    for bar, v in zip(bars, sc.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(5, n_ann_split*0.005),
                f"{v:,}\n({v/n_ann_split*100:.1f}%)",
                ha="center", va="bottom", fontsize=7)

    n_gt_s  = split_df["n_gt"].sum()
    n_seg_s = split_df["n_seg"].sum()
    ax.set_xlabel(f"GT={n_gt_s:,}  Seg={n_seg_s:,}  ({n_seg_s/max(1,n_gt_s)*100:.1f}%)")

plt.tight_layout()
plt.savefig(WORK_DIR / "eda_s55_split.png", dpi=130, bbox_inches="tight")
plt.show()

# Print table
print(f"{'Split':<10} {'n_img':>7} {'n_gt':>8} {'n_seg':>8} {'pct':>6} {'perfect':>8} {'empty':>7} {'over':>6}")
for s in ["train", "val", "test"]:
    sdf = df_cov[df_cov["split"] == s]
    if len(sdf) == 0: continue
    ng = sdf["n_gt"].sum(); ns = sdf["n_seg"].sum()
    n_perf  = (sdf["status"]=="perfect").sum()
    n_empty = (sdf["status"]=="empty").sum()
    n_over  = (sdf["status"]=="over").sum()
    print(f"{s:<10} {len(sdf):>7,} {ng:>8,} {ns:>8,} {ns/max(1,ng)*100:>5.1f}% "
          f"{n_perf:>8,} {n_empty:>7,} {n_over:>6,}")


### 5.6 — Visual Spot-Check (16 Random Samples with Masks)

In [ ]:
# 16-image grid: 4 rows (perfect / partial / empty / over) × 4 cols
def yolo_to_pixel(cx, cy, bw, bh, W, H):
    x1 = int((cx - bw/2) * W); y1 = int((cy - bh/2) * H)
    x2 = int((cx + bw/2) * W); y2 = int((cy + bh/2) * H)
    return max(0,x1), max(0,y1), min(W,x2), min(H,y2)

def draw_seg_overlay(img_rgb, seg_file, gt_boxes, img_w, img_h):
    vis = img_rgb.copy()
    cls_fill = {0: (255, 80, 0), 1: (30, 120, 255)}
    if seg_file.exists() and seg_file.stat().st_size > 0:
        for line in seg_file.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 7: continue
            try:
                cls_id = int(parts[0])
                pts = np.array(list(map(float, parts[1:]))).reshape(-1, 2)
                pts[:, 0] *= img_w; pts[:, 1] *= img_h
                pts = pts.astype(np.int32).reshape(-1, 1, 2)
                c = cls_fill.get(cls_id, (200, 200, 200))
                ov = vis.copy()
                cv2.fillPoly(ov, [pts], c)
                vis = cv2.addWeighted(vis, 0.55, ov, 0.45, 0)
                cv2.polylines(vis, [pts], True, c, 2)
            except: pass
    for cls_id, cx, cy, bw, bh in gt_boxes:
        x1,y1,x2,y2 = yolo_to_pixel(cx, cy, bw, bh, img_w, img_h)
        cv2.rectangle(vis, (x1,y1), (x2,y2), (0,255,0), 2)
    return vis

n_per_status = 4
sample_stems_by_status = {}
for s in ["perfect", "partial", "empty", "over"]:
    pool = df_cov[df_cov["status"] == s]["stem"].tolist()
    sample_stems_by_status[s] = random.sample(pool, min(n_per_status, len(pool)))

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle(
    "S5.6 — Visual Spot-Check  |  Green bbox = GT  |  Colored fill = Seg mask\n"
    "Row: perfect / partial / empty / over",
    fontsize=10)

for row_idx, s in enumerate(["perfect", "partial", "empty", "over"]):
    stems = sample_stems_by_status.get(s, [])
    for col_idx in range(4):
        ax = axes[row_idx][col_idx]
        ax.axis("off")
        if col_idx >= len(stems): continue
        stem = stems[col_idx]
        img_path = IMG_DIR / (stem + ".jpg")
        img_cv = cv2.imread(str(img_path))
        if img_cv is None: continue
        H, W = img_cv.shape[:2]
        img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
        vis = draw_seg_overlay(img_rgb, LABELS_SEG_RAW / (stem + ".txt"),
                               img_ann_map[stem], W, H)
        row_info = df_cov[df_cov["stem"] == stem].iloc[0]
        ax.imshow(vis)
        ax.set_title(f"[{s}] GT={int(row_info.n_gt)} Seg={int(row_info.n_seg)}",
                     fontsize=7, pad=2)

plt.tight_layout()
plt.savefig(WORK_DIR / "eda_s56_visual.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: eda_s56_visual.png")


## S6. Preprocessing → `labels_seg_clean`

Three steps:
1. Remove exact duplicate polygon lines within each file
2. Remove degenerate polygons (< 3 points, odd coord count, out-of-range)
3. Deduplicate image paths in train/val/test `.txt` files (prevent YOLO dup warnings)


In [ ]:
# =============================================================================
#  S6.1 — Clean Seg Labels → labels_seg_clean/
# =============================================================================
print("Cleaning seg labels...")
LABELS_SEG_CLEAN.mkdir(parents=True, exist_ok=True)

stats = {
    "files_written": 0, "lines_in": 0, "lines_out": 0,
    "removed_dup": 0, "removed_degen": 0, "removed_oor": 0,
}

for stem in tqdm(ann_stems, desc="Cleaning", ncols=80):
    seg_file = LABELS_SEG_RAW / (stem + ".txt")

    raw_lines = []
    if seg_file.exists() and seg_file.stat().st_size > 0:
        raw_lines = [l.strip() for l in seg_file.read_text().splitlines() if l.strip()]

    stats["lines_in"] += len(raw_lines)

    seen       = set()   # exact dedup
    clean_lines = []

    for line in raw_lines:
        # Step 1: exact duplicate
        if line in seen:
            stats["removed_dup"] += 1
            continue
        seen.add(line)

        parts  = line.split()
        n_tok  = len(parts)
        n_crd  = n_tok - 1

        # Step 2: degenerate (< 3 points OR odd coord count)
        if n_tok < 7 or n_crd % 2 != 0:
            stats["removed_degen"] += 1
            continue

        # Step 3: out-of-range coordinates
        try:
            coords = list(map(float, parts[1:]))
        except:
            stats["removed_degen"] += 1
            continue

        if not all(0.0 <= v <= 1.0 for v in coords):
            # Clamp instead of discard (minor overshoots at boundary)
            clamped = [round(max(0.0, min(1.0, v)), 6) for v in coords]
            try:
                cls_id = int(parts[0])
            except:
                stats["removed_oor"] += 1
                continue
            line = str(cls_id) + " " + " ".join(f"{v:.6f}" for v in clamped)
            stats["removed_oor"] += 1   # counted as fixed (still kept)

        clean_lines.append(line)

    stats["lines_out"] += len(clean_lines)
    out_path = LABELS_SEG_CLEAN / (stem + ".txt")
    out_path.write_text("\n".join(clean_lines) + ("\n" if clean_lines else ""))
    stats["files_written"] += 1

print()
print("=" * 55)
print("S6.1 — Cleaning Summary")
print("=" * 55)
print(f"  Files written          : {stats['files_written']:,}")
print(f"  Lines in (raw)         : {stats['lines_in']:,}")
print(f"  Removed exact dup      : {stats['removed_dup']:,}")
print(f"  Removed degenerate     : {stats['removed_degen']:,}")
print(f"  Coords clamped/fixed   : {stats['removed_oor']:,}")
print(f"  Lines out (clean)      : {stats['lines_out']:,}")
print(f"  Net reduction          : {stats['lines_in'] - stats['lines_out']:,}  "
      f"({(1 - stats['lines_out']/max(1,stats['lines_in']))*100:.2f}%)")


In [ ]:
# =============================================================================
#  S6.2 — Deduplicate Image Lists (prevent YOLO dup warnings)
# =============================================================================
print("\nDeduplicating image split lists...")

for list_file in [TRAIN_LIST, VAL_LIST, TEST_LIST]:
    if not list_file.exists():
        continue
    original = [l.strip() for l in open(list_file) if l.strip()]
    deduped  = list(dict.fromkeys(original))  # preserve order
    n_removed = len(original) - len(deduped)
    if n_removed > 0:
        with open(list_file, "w") as f:
            f.write("\n".join(deduped) + "\n")
        print(f"  {list_file.name}: removed {n_removed} duplicate paths")
    else:
        print(f"  {list_file.name}: no duplicates  ({len(deduped):,} paths)")

# Cross-split overlap check (images appearing in both train and val)
train_set = set(Path(l.strip()).stem for l in open(TRAIN_LIST) if l.strip())
val_set   = set(Path(l.strip()).stem for l in open(VAL_LIST)   if l.strip())
test_set  = set(Path(l.strip()).stem for l in open(TEST_LIST)  if l.strip())

tv_overlap = train_set & val_set
tt_overlap = train_set & test_set
vt_overlap = val_set   & test_set

print(f"\nCross-split overlap:")
print(f"  train ∩ val  : {len(tv_overlap):,}  {'⚠️ LEAK' if tv_overlap else '✓'}")
print(f"  train ∩ test : {len(tt_overlap):,}  {'⚠️ LEAK' if tt_overlap else '✓'}")
print(f"  val ∩ test   : {len(vt_overlap):,}  {'⚠️ LEAK' if vt_overlap else '✓'}")


In [ ]:
# =============================================================================
#  S6.3 — (Optional) BBox Rectangle Fallback for Images with No Seg Labels
# =============================================================================
if BBOX_FALLBACK_FOR_MISSING:
    print("\nS6.3 — BBox fallback for images with no seg labels...")

    def bbox_to_rect_polygon(cx, cy, bw, bh):
        """YOLO bbox → 4-corner rectangle polygon (normalized coords)."""
        x1, y1 = max(0.0, cx - bw/2), max(0.0, cy - bh/2)
        x2, y2 = min(1.0, cx + bw/2), min(1.0, cy + bh/2)
        return [x1, y1, x2, y1, x2, y2, x1, y2]

    n_fallback = 0
    for stem in tqdm(ann_stems, desc="Fallback", ncols=80):
        clean_file = LABELS_SEG_CLEAN / (stem + ".txt")
        if clean_file.exists() and clean_file.stat().st_size > 0:
            continue  # already has seg labels
        gt_boxes = img_ann_map.get(stem, [])
        if not gt_boxes:
            continue
        lines = []
        for cls_id, cx, cy, bw, bh in gt_boxes:
            pts = bbox_to_rect_polygon(cx, cy, bw, bh)
            line = str(cls_id) + " " + " ".join(f"{v:.6f}" for v in pts)
            lines.append(line)
        clean_file.write_text("\n".join(lines) + "\n")
        n_fallback += 1

    print(f"  Bbox fallback applied to {n_fallback:,} images")
else:
    print("S6.3 — BBox fallback DISABLED (BBOX_FALLBACK_FOR_MISSING=False)")


In [ ]:
# =============================================================================
#  S6.4 — Post-Clean Statistics
# =============================================================================
print("\n" + "=" * 65)
print("S6.4 — POST-CLEAN STATISTICS")
print("=" * 65)

cov_after = []
for stem in ann_stems:
    clean_file = LABELS_SEG_CLEAN / (stem + ".txt")
    gt_boxes   = img_ann_map[stem]
    n_gt       = len(gt_boxes)
    n_seg      = 0
    if clean_file.exists() and clean_file.stat().st_size > 0:
        n_seg = sum(1 for l in clean_file.read_text().splitlines() if l.strip())
    status = ("perfect" if n_seg == n_gt else
              "over"    if n_seg > n_gt  else
              "empty"   if n_seg == 0   else "partial")
    cov_after.append({"stem": stem, "n_gt": n_gt, "n_seg": n_seg, "status": status})

df_after = pd.DataFrame(cov_after)
n_tot_after = df_after["n_seg"].sum()
n_tot_gt    = df_after["n_gt"].sum()

print(f"Clean seg files  : {len(list(LABELS_SEG_CLEAN.glob('*.txt'))):,}")
print(f"Total clean lines: {n_tot_after:,}  ({n_tot_after/max(1,n_tot_gt)*100:.1f}% of GT boxes)")
print()
print("Coverage status (after cleaning):")
sc_after = df_after["status"].value_counts().reindex(STATUS_ORDER, fill_value=0)
for s, c in sc_after.items():
    diff_key = s
    before_c = df_cov["status"].value_counts().get(s, 0)
    delta = c - before_c
    delta_str = f"  (Δ{delta:+d})" if delta != 0 else ""
    print(f"  {s:<10}: {c:,}  ({c/n_ann*100:.1f}%){delta_str}")

print()
print(f"Removed by cleaning:")
print(f"  Exact duplicates   : {stats['removed_dup']:,}")
print(f"  Degenerate polygons: {stats['removed_degen']:,}")
print(f"  Total net removed  : {stats['lines_in'] - stats['lines_out']:,}")
print()
print("✓ labels_seg_clean/ ready for training.")


## S7. Build YAML + Symlinks for Seg Training

In [ ]:
# ── Symlinks ─────────────────────────────────────────────────────────────────
# /kaggle/working/images/ → actual images dir (read-only Kaggle input)
# /kaggle/working/labels/ → labels_seg_clean (our cleaned seg labels)
# YOLO derives label path: img_path.replace('/images/', '/labels/')

for link, target in [
    (WORKING_IMAGES, IMG_DIR),
    (WORKING_LABELS, LABELS_SEG_CLEAN),
]:
    if link.exists() or link.is_symlink():
        link.unlink()
    link.symlink_to(target)
    print(f"Symlink: {link.name} → {target}")

# Verify symlinks
print(f"\n  images/ → {WORKING_IMAGES.resolve()}")
print(f"  labels/ → {WORKING_LABELS.resolve()}")

# Quick sanity check: pick first train image and resolve its label
sample_line = open(TRAIN_LIST).readline().strip()
sample_img  = Path(sample_line)
sample_lbl  = Path(str(sample_img).replace("/images/", "/labels/")).with_suffix(".txt")
print(f"\nSanity check:")
print(f"  img: {sample_img}  exists={sample_img.exists()}")
print(f"  lbl: {sample_lbl}  exists={sample_lbl.exists()}")


In [ ]:
# ── data YAML ─────────────────────────────────────────────────────────────────
import yaml

data_cfg = {
    "path"  : str(WORK_DIR),
    "train" : str(TRAIN_LIST),
    "val"   : str(VAL_LIST),
    "test"  : str(TEST_LIST),
    "nc"    : 2,
    "names" : {0: "fire", 1: "smoke"},
}

with open(YAML_PATH, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)

print(f"YAML written: {YAML_PATH}")
print()
print(open(YAML_PATH).read())

# Validate first 5 label files via YAML resolution
print("Label path validation (first 5 train images):")
lines = open(TRAIN_LIST).readlines()[:5]
for line in lines:
    img_p = Path(line.strip())
    lbl_p = Path(str(img_p).replace("/images/", "/labels/")).with_suffix(".txt")
    n_lines = sum(1 for _ in open(lbl_p)) if lbl_p.exists() else 0
    print(f"  img_exists={img_p.exists()}  lbl_exists={lbl_p.exists()}  lbl_lines={n_lines}  → {lbl_p.name}")


## S8. YOLO11m-seg Resume/Fresh Training

Run **S0 → S7 first** every Kaggle session.  
If `RESUME_TRAINING=True`, this section copies the previous run directory from `/kaggle/input` into `/kaggle/working/fasdd_cv/...`, verifies `weights/last.pt`, patches `args.yaml` to point to the current `fasdd_cv_seg.yaml`, then calls:

```python
YOLO(str(last_pt)).train(resume=True)
```

Use `last.pt` for resume. Use `best.pt` only for evaluation/export.


In [ ]:
# Save run config snapshot (Layer 2) BEFORE training/resume
CONFIG_PATH = save_run_config(RUN_DIR, extra={
    "labels_seg_raw_count"  : len(list(LABELS_SEG_RAW.glob("*.txt"))),
    "labels_seg_clean_count": len(list(LABELS_SEG_CLEAN.glob("*.txt"))),
    "clean_removed_dup"     : stats["removed_dup"],
    "clean_removed_degen"   : stats["removed_degen"],
    "resume_training"       : RESUME_TRAINING,
    "resume_run_input"      : str(RESUME_RUN_INPUT) if RESUME_TRAINING else None,
    "resume_last_pt_input"  : str(RESUME_LAST_PT) if RESUME_TRAINING else None,
})


In [ ]:
from ultralytics import YOLO
import torch, shutil, os, yaml
from pathlib import Path

device_str = "0,1" if USE_MULTI_GPU and torch.cuda.device_count() > 1 else ("0" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("Training mode")
print("=" * 70)
print(f"  Resume training : {RESUME_TRAINING}")
print(f"  Device          : {device_str}")
print(f"  YAML            : {YAML_PATH}")
print(f"  Run name        : {RUN_NAME}")
print(f"  Run dir         : {RUN_DIR}")
print("=" * 70)

# Keep wandb run name aligned with our run folder.
if WANDB_ENABLED:
    os.environ["WANDB_RUN_NAME"] = RUN_NAME

def patch_args_yaml(args_yaml: Path):
    """
    Patch copied args.yaml so resumed Ultralytics training uses the current Kaggle working paths.
    This matters because /kaggle/input is read-only and each Kaggle session recreates /kaggle/working.
    """
    if not args_yaml.exists():
        print(f"args.yaml not found, skip patch: {args_yaml}")
        return

    with open(args_yaml, "r") as f:
        args = yaml.safe_load(f) or {}

    # Critical current-session paths.
    args["data"]    = str(YAML_PATH)
    args["project"] = str(PROJECT_DIR)
    args["name"]    = RUN_NAME
    args["save_dir"] = str(RUN_DIR)

    # Keep total target epochs configurable from S0.
    args["epochs"]  = EPOCHS

    # Keep hardware settings configurable from S0.
    args["batch"]   = BATCH
    args["workers"] = WORKERS
    args["device"]  = device_str

    with open(args_yaml, "w") as f:
        yaml.safe_dump(args, f, sort_keys=False)

    print(f"Patched args.yaml: {args_yaml}")
    print(f"  data    : {args.get('data')}")
    print(f"  project : {args.get('project')}")
    print(f"  name    : {args.get('name')}")
    print(f"  epochs  : {args.get('epochs')}")
    print(f"  device  : {args.get('device')}")

if RESUME_TRAINING:
    print("\n[resume] Preparing writable checkpoint directory...")

    assert RESUME_RUN_INPUT.exists(), (
        f"Resume run folder not found:\n  {RESUME_RUN_INPUT}\n"
        "Check that the Kaggle dataset is attached and the S0 path is correct."
    )
    assert RESUME_LAST_PT.exists(), (
        f"Resume last.pt not found:\n  {RESUME_LAST_PT}\n"
        "Use weights/last.pt for true resume, not best.pt."
    )

    working_last_pt = RUN_DIR / "weights" / "last.pt"
    working_args_yaml = RUN_DIR / "args.yaml"

    # Do not overwrite a more advanced working checkpoint if this cell is rerun in the same live session.
    if working_last_pt.exists():
        print(f"[resume] Existing working checkpoint found, not overwriting:")
        print(f"         {working_last_pt}")
    else:
        if RUN_DIR.exists():
            print(f"[resume] Existing RUN_DIR has no last.pt; merging input run into it:")
            print(f"         {RUN_DIR}")
        else:
            print(f"[resume] Copying input run folder:")
            print(f"         from: {RESUME_RUN_INPUT}")
            print(f"         to  : {RUN_DIR}")

        shutil.copytree(RESUME_RUN_INPUT, RUN_DIR, dirs_exist_ok=True)

    working_last_pt = RUN_DIR / "weights" / "last.pt"
    working_best_pt = RUN_DIR / "weights" / "best.pt"
    working_args_yaml = RUN_DIR / "args.yaml"

    print("\n[resume] Verification:")
    print(f"  working last.pt : {'OK' if working_last_pt.exists() else 'MISSING'} — {working_last_pt}")
    print(f"  working best.pt : {'OK' if working_best_pt.exists() else 'MISSING'} — {working_best_pt}")
    print(f"  args.yaml       : {'OK' if working_args_yaml.exists() else 'MISSING'} — {working_args_yaml}")

    if working_args_yaml.exists():
        patch_args_yaml(working_args_yaml)

    if working_last_pt.exists() and working_args_yaml.exists():
        print("\n[resume] Starting true Ultralytics resume from last.pt...")
        model = YOLO(str(working_last_pt))
        results = model.train(resume=True)
    elif working_last_pt.exists():
        print("\n[resume] WARNING: args.yaml missing.")
        print("         Falling back to warm-start from last.pt with fresh optimizer state.")
        print("         This is NOT a true optimizer/LR resume.")
        model = YOLO(str(working_last_pt))
        results = model.train(
            data          = str(YAML_PATH),
            epochs        = EPOCHS,
            imgsz         = IMGSZ,
            batch         = BATCH,
            workers       = WORKERS,
            lr0           = LR0,
            lrf           = LRF,
            device        = device_str,
            project       = str(PROJECT_DIR),
            name          = RUN_NAME + "_warmstart",
            hsv_h         = AUGMENT_HSV_H,
            hsv_s         = AUGMENT_HSV_S,
            hsv_v         = AUGMENT_HSV_V,
            flipud        = AUGMENT_FLIPUD,
            degrees       = AUGMENT_DEGREES,
            translate     = AUGMENT_TRANSLATE,
            scale         = AUGMENT_SCALE,
            patience      = 50,
            warmup_epochs = 3.0,
            mosaic        = 1.0,
            weight_decay  = 0.0005,
            verbose       = True,
            seed          = SEED,
        )
    else:
        raise FileNotFoundError(f"Could not prepare working last.pt: {working_last_pt}")

else:
    print("\n[fresh] Starting fresh finetune from SEG_MODEL_INIT...")
    model = YOLO(SEG_MODEL_INIT)
    results = model.train(
        data          = str(YAML_PATH),
        epochs        = EPOCHS,
        imgsz         = IMGSZ,
        batch         = BATCH,
        workers       = WORKERS,
        lr0           = LR0,
        lrf           = LRF,
        device        = device_str,
        project       = str(PROJECT_DIR),
        name          = RUN_NAME,
        # Augmentation (EDA-derived — same as detection baseline)
        hsv_h         = AUGMENT_HSV_H,
        hsv_s         = AUGMENT_HSV_S,
        hsv_v         = AUGMENT_HSV_V,
        flipud        = AUGMENT_FLIPUD,
        degrees       = AUGMENT_DEGREES,
        translate     = AUGMENT_TRANSLATE,
        scale         = AUGMENT_SCALE,
        # Training schedule
        patience      = 50,
        warmup_epochs = 3.0,
        mosaic        = 1.0,
        weight_decay  = 0.0005,
        verbose       = True,
        seed          = SEED,
    )

print(f"\nDone. Save dir  : {results.save_dir}")
print(f"Best model: {results.save_dir}/weights/best.pt")
print(f"Last model: {results.save_dir}/weights/last.pt")
if WANDB_ENABLED and wandb.run:
    print(f"wandb URL : {wandb.run.get_url()}")


## S9. Post-Training Logging

In [ ]:
try:
    active_results = results
except NameError:
    print("No training results — run S8 first.")
    active_results = None

if active_results is not None:
    rd = active_results.results_dict

    def safe(key):
        v = rd.get(key)
        return round(float(v), 4) if v is not None else None

    # Box head
    box_mAP50   = safe("metrics/mAP50(B)")
    box_mAP5095 = safe("metrics/mAP50-95(B)")
    precision   = safe("metrics/precision(B)")
    recall      = safe("metrics/recall(B)")

    # Seg head
    seg_mAP50   = safe("metrics/mAP50(M)")
    seg_mAP5095 = safe("metrics/mAP50-95(M)")

    # Per-class AP
    box_ap50_fire  = box_ap50_smoke  = None
    seg_ap50_fire  = seg_ap50_smoke  = None
    try:
        box = active_results.box
        if hasattr(box, "ap50") and len(box.ap50) >= 2:
            box_ap50_fire  = round(float(box.ap50[0]), 4)
            box_ap50_smoke = round(float(box.ap50[1]), 4)
    except Exception as e:
        print(f"Box per-class AP: {e}")

    try:
        seg = active_results.seg
        if hasattr(seg, "ap50") and len(seg.ap50) >= 2:
            seg_ap50_fire  = round(float(seg.ap50[0]), 4)
            seg_ap50_smoke = round(float(seg.ap50[1]), 4)
    except Exception as e:
        print(f"Seg per-class AP: {e}")

    ap_gap = (round(box_ap50_smoke - box_ap50_fire, 4)
              if box_ap50_fire is not None and box_ap50_smoke is not None else None)
    epochs_run = getattr(active_results, "epoch", None)

    wandb_url = None
    if WANDB_ENABLED:
        try: wandb_url = wandb.run.get_url()
        except: pass

    print("=" * 60)
    print("Post-training results:")
    print(f"  Box mAP@0.5        : {box_mAP50}")
    print(f"  Box mAP@0.5:0.95   : {box_mAP5095}")
    print(f"  Box AP50 fire      : {box_ap50_fire}")
    print(f"  Box AP50 smoke     : {box_ap50_smoke}")
    print(f"  Box AP gap (s-f)   : {ap_gap}")
    print(f"  Seg mAP@0.5        : {seg_mAP50}")
    print(f"  Seg mAP@0.5:0.95   : {seg_mAP5095}")
    print(f"  Seg AP50 fire      : {seg_ap50_fire}")
    print(f"  Seg AP50 smoke     : {seg_ap50_smoke}")
    print(f"  Precision          : {precision}")
    print(f"  Recall             : {recall}")
    print(f"  Epochs run         : {epochs_run}")
    print("=" * 60)


In [ ]:
# ── Layer 2: update config snapshot ───────────────────────────────────────────
if active_results is not None:
    cfg_file = RUN_DIR / "run_config.json"
    if cfg_file.exists():
        with open(cfg_file) as f:
            cfg = _json.load(f)
    else:
        cfg = {}
    cfg.setdefault("results", {}).update({
        "box_mAP50": box_mAP50,       "box_mAP50_fire": box_ap50_fire,
        "box_mAP50_smoke": box_ap50_smoke, "box_mAP5095": box_mAP5095,
        "seg_mAP50": seg_mAP50,       "seg_mAP50_fire": seg_ap50_fire,
        "seg_mAP50_smoke": seg_ap50_smoke, "seg_mAP5095": seg_mAP5095,
        "AP_gap": ap_gap, "epochs_run": epochs_run, "wandb_url": wandb_url,
    })
    with open(cfg_file, "w") as f:
        _json.dump(cfg, f, indent=2)
    print(f"Layer 2 updated: {cfg_file}")

    # ── Layer 3: append to registry ──────────────────────────────────────────
    row = {
        "run_name"                 : RUN_NAME,
        "date"                     : DATE_STR,
        "model"                    : f"yolo11{YOLO_SIZE}-seg",
        "mode"                     : TRAIN_MODE,
        "imgsz"                    : IMGSZ,
        "epochs_run"               : epochs_run,
        "box_mAP50"                : box_mAP50,
        "box_mAP50_fire"           : box_ap50_fire,
        "box_mAP50_smoke"          : box_ap50_smoke,
        "box_mAP5095"              : box_mAP5095,
        "seg_mAP50"                : seg_mAP50,
        "seg_mAP50_fire"           : seg_ap50_fire,
        "seg_mAP50_smoke"          : seg_ap50_smoke,
        "seg_mAP5095"              : seg_mAP5095,
        "AP_gap_smoke_minus_fire_box": ap_gap,
        "precision"                : precision,
        "recall"                   : recall,
        "notes"                    : EXPERIMENT_NOTES,
        "wandb_run_url"            : wandb_url,
    }
    df_reg = append_to_registry(row)

    # ── Layer 1: log to wandb ─────────────────────────────────────────────────
    if WANDB_ENABLED and wandb.run:
        wandb.summary.update({
            "test/box_mAP50"  : box_mAP50,
            "test/seg_mAP50"  : seg_mAP50,
            "test/box_AP_fire": box_ap50_fire,
            "test/seg_AP_fire": seg_ap50_fire,
            "test/AP_gap"     : ap_gap,
        })
        wandb.finish()
        print(f"wandb finished: {wandb_url}")

    print("\nCurrent experiment registry:")
    show_registry()


## S10. Evaluation on Official Test Split

In [ ]:
from ultralytics import YOLO

best_pt = None
for candidate in [
    RUN_DIR / "weights" / "best.pt",
    PROJECT_DIR / RUN_NAME / "weights" / "best.pt",
    RESUME_BEST_PT if RESUME_TRAINING else Path("__no_resume_best__.pt"),
]:
    if candidate.exists():
        best_pt = candidate
        break

if best_pt is None:
    print("best.pt not found — run S8 first.")
else:
    print(f"Evaluating: {best_pt}")
    eval_model = YOLO(str(best_pt))

    metrics = eval_model.val(
        data    = str(YAML_PATH),
        split   = "test",
        imgsz   = IMGSZ,
        batch   = 32,
        device  = "0",
        verbose = True,
    )

    print("\n=== Test Split Results ===")
    print(f"Box mAP@0.5       : {metrics.box.map50:.4f}")
    print(f"Box mAP@0.5:0.95  : {metrics.box.map:.4f}")
    print(f"Box Precision     : {metrics.box.mp:.4f}")
    print(f"Box Recall        : {metrics.box.mr:.4f}")

    try:
        print(f"Seg mAP@0.5       : {metrics.seg.map50:.4f}")
        print(f"Seg mAP@0.5:0.95  : {metrics.seg.map:.4f}")
    except:
        print("Seg metrics not available")

    cls_names = {0: "fire", 1: "smoke"}
    print("\nPer-class Box AP@0.5:")
    for i, name in cls_names.items():
        if i < len(metrics.box.ap50):
            print(f"  {name:5s}: {metrics.box.ap50[i]:.4f}")
    if len(metrics.box.ap50) >= 2:
        gap = metrics.box.ap50[1] - metrics.box.ap50[0]
        print(f"  AP gap (smoke-fire): {gap:+.4f}")

    try:
        print("\nPer-class Seg AP@0.5:")
        for i, name in cls_names.items():
            if i < len(metrics.seg.ap50):
                print(f"  {name:5s}: {metrics.seg.ap50[i]:.4f}")
    except:
        pass


In [ ]:
# Training curves from results.csv
if best_pt is not None:
    run_dir = best_pt.parent.parent
    results_csv = run_dir / "results.csv"
    if results_csv.exists():
        df_res = pd.read_csv(results_csv)
        df_res.columns = [c.strip() for c in df_res.columns]

        def find_col(candidates):
            for c in candidates:
                if c in df_res.columns: return c
            return None

        plots = [
            (["train/box_loss"],                          "Train Box Loss"),
            (["train/seg_loss"],                          "Train Seg Loss"),
            (["train/cls_loss"],                          "Train Cls Loss"),
            (["train/dfl_loss"],                          "Train DFL Loss"),
            (["metrics/mAP50(B)", "metrics/mAP_0.5"],    "Box mAP@0.5"),
            (["metrics/mAP50(M)"],                        "Seg mAP@0.5"),
        ]

        fig, axes = plt.subplots(2, 3, figsize=(17, 8))
        fig.suptitle(f"Training Curves — {RUN_NAME}", fontsize=11, fontweight="bold")
        for ax, (candidates, title) in zip(axes.ravel(), plots):
            col = find_col(candidates)
            if col:
                ax.plot(df_res.index, df_res[col], color="#3498db", lw=2)
            ax.set_title(title, fontsize=9)
            ax.set_xlabel("Epoch")
            ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(WORK_DIR / "training_curves.png", dpi=130, bbox_inches="tight")
        plt.show()
    else:
        print(f"results.csv not found at {results_csv}")


## S11. Inference Spot-Check

In [ ]:
import time

if best_pt is None:
    print("Run S8 first.")
else:
    infer_model = YOLO(str(best_pt))
    test_paths  = [p.strip() for p in open(TEST_LIST)]
    CONF, IOU   = 0.25, 0.45

    # ── Latency benchmark ───────────────────────────────────────────────────
    N_WARMUP, N_BENCH = 5, 20
    bench_paths = random.sample(test_paths, N_WARMUP + N_BENCH)
    for p in bench_paths[:N_WARMUP]:
        infer_model.predict(p, imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)
    t0 = time.perf_counter()
    for p in bench_paths[N_WARMUP:]:
        infer_model.predict(p, imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)
    elapsed = time.perf_counter() - t0
    inf_ms  = elapsed / N_BENCH * 1000
    fps     = 1000 / inf_ms
    model_mb = round(best_pt.stat().st_size / 1e6, 1)
    print(f"Inference latency : {inf_ms:.1f} ms/img  ({fps:.1f} FPS)  Model: {model_mb} MB")

    # Update Layer 2 + 3
    if (RUN_DIR / "run_config.json").exists():
        with open(RUN_DIR / "run_config.json") as f: cfg = _json.load(f)
        cfg["results"]["inference_ms"] = round(inf_ms, 1)
        cfg["results"]["model_mb"]     = model_mb
        with open(RUN_DIR / "run_config.json", "w") as f: _json.dump(cfg, f, indent=2)
    df_reg = load_registry()
    mask_r = df_reg["run_name"] == RUN_NAME
    if mask_r.any():
        df_reg.loc[mask_r, "inference_ms_t4"] = round(inf_ms, 1)
        df_reg.loc[mask_r, "fps_t4"]          = round(fps, 1)
        df_reg.loc[mask_r, "model_mb"]        = model_mb
        df_reg.to_csv(REGISTRY_PATH, index=False)
    show_registry()

    # ── Visual spot-check 4×4 grid ──────────────────────────────────────────
    sample_paths = random.sample(test_paths, 16)
    cls_colors   = {0: (255, 80, 0), 1: (30, 120, 255)}
    cls_names_d  = {0: "fire", 1: "smoke"}

    fig, axes = plt.subplots(4, 4, figsize=(16, 14))
    fig.suptitle(f"Inference Spot-Check — conf={CONF}  imgsz={IMGSZ}", fontsize=11)
    axes = axes.ravel()

    for i, img_path in enumerate(sample_paths):
        ax = axes[i]
        img_bgr = cv2.imread(img_path)
        if img_bgr is None: ax.axis("off"); continue
        H, W  = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        r = infer_model.predict(img_path, conf=CONF, iou=IOU, imgsz=IMGSZ, verbose=False)
        vis = img_rgb.copy()

        # Draw seg masks
        if r[0].masks is not None:
            for mask_data, cls_id in zip(r[0].masks.data, r[0].boxes.cls):
                m = mask_data.cpu().numpy().astype(np.uint8)
                m_resized = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
                c = cls_colors.get(int(cls_id), (200,200,200))
                ov = vis.copy()
                ov[m_resized > 0] = np.array(c, dtype=np.uint8)
                vis = cv2.addWeighted(vis, 0.55, ov, 0.45, 0)

        # Draw bboxes
        for box in r[0].boxes:
            xy   = box.xyxy[0].cpu().numpy().astype(int)
            conf = float(box.conf[0])
            cls  = int(box.cls[0])
            c    = cls_colors.get(cls, (200,200,200))
            cv2.rectangle(vis, (xy[0],xy[1]), (xy[2],xy[3]), c, 2)
            cv2.putText(vis, f"{cls_names_d.get(cls,'?')} {conf:.2f}",
                        (xy[0], max(xy[1]-4, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, c, 1)

        ax.imshow(vis)
        ax.set_title(Path(img_path).name[:22], fontsize=6)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(WORK_DIR / "inference_spotcheck.png", dpi=130, bbox_inches="tight")
    plt.show()


## S12. Save Checkpoints

Run before session expires to preserve weights.


In [ ]:
import subprocess

SAVE_DIR = PROJECT_DIR
ZIP_OUT  = WORK_DIR / "fasdd_cv_seg.zip"

if SAVE_DIR.exists():
    runs = sorted(SAVE_DIR.iterdir())
    print(f"Found {len(runs)} run(s) in {SAVE_DIR}:")
    for run in runs:
        weights = run / "weights"
        if weights.exists():
            best    = weights / "best.pt"
            last    = weights / "last.pt"
            best_mb = round(best.stat().st_size / 1e6, 1) if best.exists() else "missing"
            last_mb = round(last.stat().st_size / 1e6, 1) if last.exists() else "missing"
            print(f"  {run.name}: best={best_mb} MB  last={last_mb} MB")

    print(f"\nCreating {ZIP_OUT} ...")
    subprocess.run(
        ["zip", "-r", str(ZIP_OUT), "fasdd_cv"],
        cwd=str(WORK_DIR), check=True,
    )
    zip_mb = round(ZIP_OUT.stat().st_size / 1e6, 1)
    print(f"Done: fasdd_cv_seg.zip ({zip_mb} MB) — download from Output tab")
else:
    print(f"Nothing to save: {SAVE_DIR} does not exist.")
